# Phase 2 — batched inference

How does throughput / latency / memory change as we process N requests together instead of one at a time?

**Sweep:** Llama-3.1-8B-Instruct, A100, two context lengths (8k & 32k), batch sizes 1 / 2 / 4 / 8 / 16. Same prompt tiled N times — all requests have the same length, which is the simplest batched workload (no padding, no dynamic scheduling).

**Output file:** `results/phase2_llama31_a100.jsonl` — same row schema as Phase 1, plus a `batch_size` column that's now varying.

**Metrics interpretation:**
- `tokens_per_second` is **aggregate** across the batch (output_tokens summed). Per-request throughput is `tokens_per_second / batch_size`.
- `ttft_seconds` is the time until the FIRST token of the batch arrives — proxy for "first user's TTFT under load."
- `tpot_seconds` is total_latency / output_tokens, also aggregate.

**OOM is data**, not a crash. Higher batch × larger context = more KV cache; we'll find the wall.


In [ ]:
# 1. Setup.
import os
os.chdir('/content/LLM_Inference')
!pip install -q -r requirements.txt
print('cwd:', os.getcwd())


In [ ]:
# 2. Hugging Face login (Llama-3.1 is gated).
from huggingface_hub import login
login()


In [ ]:
# 3. Smoke test: one (ctx, batch) cell to verify wiring before the long sweep.
!python3 scripts/run_batch_experiment.py --config configs/baseline_hf.yaml \
  --model-config configs/llama3_1_8b_instruct.yaml \
  --context-lengths 4096 \
  --batch-sizes 2 \
  --max-new-tokens 32 \
  --results-path results/phase2_smoke.jsonl

import json
for line in open('results/phase2_smoke.jsonl'):
    r = json.loads(line)
    print(f"ctx={r['context_length']} batch={r['batch_size']} "
          f"success={r['success']} ttft={r['ttft_seconds']} "
          f"agg_tps={r['tokens_per_second']} peak={r['peak_gpu_memory_gb']:.2f}GB")


## Full sweep

The full sweep is one cell, ~20–40 min on A100. Higher batch × longer context will OOM; those rows are recorded as `success=false` and the sweep continues.


In [ ]:
# 4. Sweep batch sizes 1/2/4/8/16 at ctx 8192 and 32768.
!python3 scripts/run_batch_experiment.py --config configs/baseline_hf.yaml \
  --model-config configs/llama3_1_8b_instruct.yaml \
  --context-lengths 8192 32768 \
  --batch-sizes 1 2 4 8 16 \
  --max-new-tokens 64 \
  --results-path results/phase2_llama31_a100.jsonl


## Visualizations

Loads `results/phase2_*.jsonl` and produces:
1. Per-context metric grid (TTFT, TPOT, aggregate throughput, per-request throughput, peak memory, KV-cache estimate vs batch size).
2. Latency-vs-throughput Pareto — the headline plot for "single-user latency vs system throughput."
3. Max-feasible-batch matrix (rows = context length, cols = batch size, green = OK, red = OOM).


In [ ]:
# 5. Load Phase-2 JSONLs into a DataFrame.
import json, glob
from pathlib import Path
import pandas as pd

rows = []
for path in sorted(glob.glob('results/phase2_*.jsonl')):
    if 'smoke' in path:  # don't include the smoke-test row in the analysis
        continue
    for line in open(path):
        r = json.loads(line)
        r['_source'] = Path(path).name
        rows.append(r)
if not rows:
    raise SystemExit('No results/phase2_*.jsonl files found yet — run the sweep first.')

df = pd.DataFrame(rows)
df['per_request_tps'] = df.apply(
    lambda r: (r['tokens_per_second'] / r['batch_size']) if r['success'] else None, axis=1
)
df['per_request_latency'] = df['total_latency_seconds']  # all reqs in batch finish together

print(f'loaded {len(df)} rows from {df["_source"].nunique()} files')
print(df[['model_name', 'hardware', 'context_length', 'batch_size', 'success',
          'ttft_seconds', 'total_latency_seconds',
          'tokens_per_second', 'per_request_tps',
          'peak_gpu_memory_gb', 'kv_cache_memory_gb']]
      .sort_values(['context_length', 'batch_size'])
      .to_string(index=False))


In [ ]:
# 6. Per-context metric grid: 2x3 panels, one figure per context length.
import matplotlib.pyplot as plt

PLOT_DIR = Path('results/plots'); PLOT_DIR.mkdir(parents=True, exist_ok=True)

METRICS = [
    ('ttft_seconds',          'TTFT (s)'),
    ('tpot_seconds',          'TPOT (s)'),
    ('tokens_per_second',     'Aggregate throughput (tok/s)'),
    ('per_request_tps',       'Per-request throughput (tok/s)'),
    ('peak_gpu_memory_gb',    'Peak GPU memory (GB)'),
    ('kv_cache_memory_gb',    'KV-cache estimate (GB)'),
]

for ctx, sub in df.groupby('context_length'):
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'Phase 2 — context = {ctx} tokens', fontsize=14, fontweight='bold')
    for (col, ylabel), ax in zip(METRICS, axes.flat):
        ok   = sub[sub.success == True ].sort_values('batch_size')
        fail = sub[sub.success == False].sort_values('batch_size')
        ax.plot(ok.batch_size, ok[col], marker='o')
        if len(fail):
            y = fail[col].fillna(0) if col in ('peak_gpu_memory_gb', 'kv_cache_memory_gb') else [0]*len(fail)
            ax.scatter(fail.batch_size, y, marker='x', color='red', s=120, zorder=5, linewidths=2.5)
        ax.set_xscale('log', base=2)
        ax.set_xlabel('batch size')
        ax.set_ylabel(ylabel)
        ax.grid(alpha=0.3)
    plt.tight_layout()
    out = PLOT_DIR / f'phase2_ctx{ctx}.png'
    plt.savefig(out, dpi=120, bbox_inches='tight')
    print(f'saved {out}')
    plt.show()


In [ ]:
# 7. Latency-vs-throughput Pareto.
#    Each point is one (context, batch) cell. The headline plot for "single-user
#    latency vs system throughput" — moving up-and-right is strictly better.
import matplotlib.pyplot as plt

ok = df[df.success == True].sort_values(['context_length', 'batch_size'])
fig, ax = plt.subplots(figsize=(8, 6))

for ctx, grp in ok.groupby('context_length'):
    ax.plot(grp.tokens_per_second, grp.total_latency_seconds, marker='o',
            label=f'ctx {ctx}')
    for _, row in grp.iterrows():
        ax.annotate(f'B={int(row.batch_size)}',
                    (row.tokens_per_second, row.total_latency_seconds),
                    textcoords='offset points', xytext=(6, 4), fontsize=9)

ax.set_xlabel('Aggregate throughput (tokens/sec)  →  better')
ax.set_ylabel('Per-request latency (s)  →  worse')
ax.set_title('Phase 2 — latency vs throughput (single-user vs system tradeoff)')
ax.grid(alpha=0.3)
ax.legend(title='context length')

out = Path('results/plots/phase2_pareto.png')
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()


In [ ]:
# 8. Max-feasible-batch matrix: rows = context, cols = batch, green=OK, red=OOM.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

pivot = df.pivot_table(index='context_length', columns='batch_size',
                       values='success', aggfunc='first')
fig, ax = plt.subplots(figsize=(1.2 * len(pivot.columns) + 2, 0.9 * len(pivot.index) + 1))
for i, ctx in enumerate(pivot.index):
    for j, bsz in enumerate(pivot.columns):
        v = pivot.loc[ctx, bsz]
        color = '#bde0a8' if v == True else ('#f4a8a8' if v == False else '#dddddd')
        ax.add_patch(Rectangle((j, i), 1, 1, facecolor=color, edgecolor='white'))
        label = 'OK' if v == True else ('OOM' if v == False else '-')
        ax.text(j + 0.5, i + 0.5, label, ha='center', va='center', fontsize=10)

ax.set_xlim(0, len(pivot.columns)); ax.set_ylim(0, len(pivot.index))
ax.invert_yaxis()
ax.set_xticks(np.arange(len(pivot.columns)) + 0.5)
ax.set_xticklabels(pivot.columns)
ax.set_yticks(np.arange(len(pivot.index)) + 0.5)
ax.set_yticklabels(pivot.index)
ax.set_xlabel('batch size'); ax.set_ylabel('context length (tokens)')
ax.set_title('Phase 2 — max feasible (context, batch) on A100')
ax.tick_params(length=0)
for spine in ax.spines.values(): spine.set_visible(False)
plt.tight_layout()
out = Path('results/plots/phase2_frontier.png')
plt.savefig(out, dpi=120, bbox_inches='tight')
print(f'saved {out}')
plt.show()
